In [6]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim

In [7]:
train_df = pd.read_csv("data/booking_reviews_train.csv")
test_df = pd.read_csv("data/booking_reviews_test.csv")

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train = vectorizer.fit_transform(train_df["text_combined"])
X_test = vectorizer.transform(test_df["text_combined"])

y_train = train_df["is_positive"].values
y_test = test_df["is_positive"].values

X_train = X_train.toarray()
X_test = X_test.toarray()

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [8]:
class SentimentNN(nn.Module):
    def __init__(self, input_dim):
        super(SentimentNN, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

In [9]:
input_dim = X_train.shape[1]

model = SentimentNN(input_dim)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 12
batch_size = 64

for epoch in range(epochs):
    permutation = torch.randperm(X_train.size(0))
    
    epoch_loss = 0
    
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_X = X_train[indices]
        batch_y = y_train[indices]
        
        optimizer.zero_grad()
        
        outputs = model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

Epoch 1, Loss: 128.7330
Epoch 2, Loss: 78.0014
Epoch 3, Loss: 58.6807
Epoch 4, Loss: 44.8787
Epoch 5, Loss: 34.8844
Epoch 6, Loss: 29.2016
Epoch 7, Loss: 26.1118
Epoch 8, Loss: 24.0614
Epoch 9, Loss: 23.1618
Epoch 10, Loss: 22.8713
Epoch 11, Loss: 22.3148
Epoch 12, Loss: 22.2442


In [10]:
outputs = model(X_test)
preds = (outputs >= 0.5).int()

with torch.no_grad():
    outputs = model(X_test).squeeze()
    preds = (outputs >= 0.5).int().numpy()

y_true = y_test.numpy()

In [11]:
print("Accuracy:", accuracy_score(y_true, preds))
print("Precision:", precision_score(y_true, preds))
print("Recall:", recall_score(y_true, preds))
print("F1 Score:", f1_score(y_true, preds))
print("Confusion matrix:")
print(confusion_matrix(y_true, preds))
y_scores = outputs.detach().numpy()
print("ROC AUC Score:", round(roc_auc_score(y_true, y_scores), 3))

Accuracy: 0.8305730059013897
Precision: 0.8777692895339955
Recall: 0.8936997666580244
F1 Score: 0.8856628982528263
Confusion matrix:
[[ 916  480]
 [ 410 3447]]
ROC AUC Score: 0.891
